# PathFinder: Frederick sidewalk detection pilot (Tile2Net)

Runs [Tile2Net](https://github.com/VIDA-NYU/tile2net) (BSD-3) on Maryland's public six-inch aerial imagery (MD iMAP, 2023–25) for three 1 km cells of Frederick. Only public imagery and the cells' bounds are used here; the city's survey and the scoring stay on the PathFinder laptop.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. It takes roughly 20–40 minutes. The last cell downloads `frederick_pilot_results.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!git clone --depth 1 https://github.com/VIDA-NYU/tile2net.git /content/tile2net
!pip install -e /content/tile2net


Restart is not needed. If the import below fails, use Runtime → Restart session, then run from this cell on.

In [ ]:
import json, time, io, math, os, zipfile, pathlib
import numpy as np, requests
from PIL import Image
from tile2net import Raster
CELLS = json.loads('''[
 {
  "name": "frederick_363_196",
  "south": 39.431529,
  "west": -77.429802,
  "north": 39.440578,
  "east": -77.418132
 },
 {
  "name": "frederick_366_194",
  "south": 39.413637,
  "west": -77.394853,
  "north": 39.422682,
  "east": -77.383191
 },
 {
  "name": "frederick_366_198",
  "south": 39.449665,
  "west": -77.395054,
  "north": 39.45871,
  "east": -77.383386
 }
]''')
IMG = 'https://mdgeodata.md.gov/imagery/rest/services/SixInch/SixInchImagery2023_2024_2025/MapServer/export'
ZOOM = 19
ROOT = pathlib.Path('/content/pilot')
ROOT.mkdir(exist_ok=True)

def tile_bounds_3857(x, y, z):
    n = 2 ** z; size = 2 * 20037508.342789244 / n
    xmin = -20037508.342789244 + x * size; ymax = 20037508.342789244 - y * size
    return xmin, ymax - size, xmin + size, ymax

def fetch(x, y, z, path, tries=6):
    b = tile_bounds_3857(x, y, z)
    p = {'bbox': ','.join(f'{v:.3f}' for v in b), 'bboxSR': 3857, 'imageSR': 3857,
         'size': '256,256', 'format': 'png24', 'transparent': 'false', 'f': 'image'}
    for k in range(tries):
        try:
            im = Image.open(io.BytesIO(requests.get(IMG, params=p, timeout=60).content)).convert('RGB')
            if np.asarray(im).std() > 8:  # the service sometimes returns blank tiles
                im.save(path); return True
        except Exception as exc:
            print('retry', x, y, exc)
        time.sleep(2 + 2 * k)
    return False


## 1. Download each cell's zoom-19 tiles from MD iMAP

In [ ]:
rasters = {}
for c in CELLS:
    loc = f"{c['north']}, {c['west']}, {c['south']}, {c['east']}"
    probe = Raster(location=loc, name=c['name'], zoom=ZOOM, output_dir=str(ROOT / 'out'))
    tiles_dir = ROOT / 'tiles' / c['name'] / str(ZOOM)
    todo = [(t.xtile, t.ytile) for t in np.asarray(probe.tiles).ravel()]
    missing = []
    for x, y in todo:
        path = tiles_dir / str(x) / f'{y}.png'
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists() and not fetch(x, y, ZOOM, path):
            missing.append((x, y))
    print(c['name'], len(todo), 'tiles,', len(missing), 'could not be fetched')
    rasters[c['name']] = (loc, tiles_dir)


## 2. Stitch and run the sidewalk model

In [ ]:
for name, (loc, tiles_dir) in rasters.items():
    r = Raster(location=loc, name=name, zoom=ZOOM, output_dir=str(ROOT / 'out'),
               input_dir=str(tiles_dir.parent / str(ZOOM) / 'x' / 'y.png'))
    r.generate(4)
    r.inference()
    print('done', name)


## 3. Download the results

Only the vector outputs (polygons and network) are zipped, not the imagery.

In [ ]:
keep = {'.shp', '.shx', '.dbf', '.prj', '.cpg', '.geojson', '.gpkg', '.feather', '.parquet', '.json'}
out = ROOT / 'frederick_pilot_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in (ROOT / 'out').rglob('*'):
        if f.is_file() and f.suffix.lower() in keep and 'tiles' not in f.parts:
            z.write(f, f.relative_to(ROOT / 'out'))
print(out, round(out.stat().st_size / 1e6, 1), 'MB')
from google.colab import files
files.download(str(out))
